# 10. Быстрый axial-решатель полного события

Строим настоящий сквозной расчёт: G4-подобные элементы → транспортный кэш →
редкий axial-источник → совместное Numba-применение порядков 1 и ≥2 → точная
баллистика → временные бины. Математическая постановка прежняя; меняются
порядок вычислений и место хранения промежуточных величин.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from lighthit import SolverSettings, synthetic_medium, __version__
from lighthit.cache import CacheGrid, ResponseCache
from lighthit.readout import inverse_bins
from lighthit.viewer import SCHEMA, write_event_viewer
from lighthit.experimental.g4_source import LightElements,SourceContract
from lighthit.experimental.axial_source import AxisFrame
from lighthit.experimental.axial_fast import PreparedAxialKernel,compile_axial_source_fast
from lighthit.experimental.ballistic_fast import vectorised_ballistic_bins_fast


## 10.1. Небольшой воспроизводимый ливень и массив ОМ

Производственный запуск читает миллионы элементов. Для учебного исполнения
создаём те же поля контракта, но 2500 элементов. Это проверка вычислительной
цепочки, не физическая модель среднего ливня.


In [ ]:
rng=np.random.default_rng(20260919);n=2500
z=np.sort(rng.uniform(0,10,n))
start=np.column_stack((15+rng.normal(0,.08,n),10+rng.normal(0,.08,n),-5+z))
direction=np.column_stack((rng.normal(0,.03,n),rng.normal(0,.03,n),np.ones(n)))
direction/=np.linalg.norm(direction,axis=1)[:,None]
length=rng.uniform(.003,.025,n);photons=rng.gamma(2,10,n);t=z/.299792458
contract=SourceContract()
elements=LightElements(start,direction,length,photons,np.full(n,.75),t,
    t+length/.299792458,np.arange(n),np.arange(n),contract,{"synthetic_course":True})
heights=np.linspace(-75,75,12)
receivers=np.concatenate([np.column_stack((np.zeros(12),np.zeros(12),heights)),
    np.column_stack((np.full(12,60.),np.zeros(12),heights))])
cluster=np.repeat([0,1],12);string=np.zeros(24,int);module=np.tile(np.arange(12),2)
print(elements.summary(),receivers.shape)


## 10.2. Общий транспорт и сжатый источник

Кэш хранит радиальные мультиполи среды и не зависит от события. Axial-источник
хранит $S_{\ell m}(\mathbf x_b,\omega)$. В быстром применении сплайн,
гармоника и вклад ячейки сразу поступают в аккумулятор; массив
`(ОМ, ячейка, канал, частота)` не создаётся.


In [ ]:
medium=synthetic_medium();omega=np.linspace(0,.12,11);degree=8
cache=ResponseCache.build(medium,SolverSettings(12,degree,3,.08,6),
    CacheGrid.geometric(3,180,36,omega),angular_backend="numba",radial_phase="flight")
source=compile_axial_source_fast(elements,degree,omega,azimuthal_degree=2,
    cell_m=.5,frame=AxisFrame.of(elements))
prepared=PreparedAxialKernel.from_cache(cache,degree=degree)
scattered=prepared.apply(source,receivers,source_omega_per_ns=omega,receiver_block=4)
print("source",source.channels.shape,source.channels.nbytes/2**20,"MiB; response",scattered.shape)


## 10.3. Временная система и баллистика

При соглашении $\widetilde K=\int K(t)e^{+i\omega t}dt$ переход к локальному
времени $t-t_d$ требует $e^{-i\omega t_d}$. Баллистический порядок не
Fourier-инвертируется: корень конуса и точное время прихода кладутся в бин.


In [ ]:
first=np.argmin(elements.start_ns)
origins=elements.start_ns[first]+np.linalg.norm(receivers-elements.start_m[first],axis=1)/medium.speed_m_per_ns
edges=np.arange(-40,281,20.)
ballistic,ballistic_bins=vectorised_ballistic_bins_fast(
    elements,receivers,medium,elements.cone_cosine,origins,edges)
relative=scattered*np.exp(-1j*omega[:,None]*origins[None,:])[:,:,None]
components=np.zeros((len(receivers),len(edges)-1,3));components[:,:,0]=ballistic_bins
for order in range(2):components[:,:,order+1]=inverse_bins(omega,relative[:,:,order],edges)
charge=np.column_stack((ballistic,scattered[0].real))
print("integrated",charge.sum(axis=0),"window",components.sum(axis=(0,1)))


## 10.4. Графики и переносимый результат

Стационарный заряд берётся из ω=0 и не подменяется суммой конечного временного
окна. Поэтому видны обе величины. Знаковые бины сохранены.


In [ ]:
centres=(edges[:-1]+edges[1:])/2;bright=np.argmax(np.abs(charge.sum(axis=1)))
fig,ax=plt.subplots(1,2,figsize=(11,4))
for p,label in enumerate(("0","1","≥2")):ax[0].plot(receivers[:,2],charge[:,p],".",label=label)
ax[0].set(xlabel="z ОМ, м",ylabel="интегральный отклик, м⁻²");ax[0].legend()
for p,label in enumerate(("0","1","≥2")):ax[1].step(centres,components[bright,:,p],where="mid",label=label)
ax[1].set(xlabel="время от фронта, нс",ylabel="отклик / бин",title=f"ОМ {bright}")
ax[1].legend();fig.tight_layout()


In [ ]:
payload={"schema":SCHEMA,"package_version":__version__,
"units":{"position":"m","time":"ns","signal":"photons / m² effective area"},
"detector":{"label":"Учебный массив","positions_m":receivers.tolist(),
"cluster_id":cluster.tolist(),"string_id":string.tolist(),"module_id":module.tolist()},
"medium":{"provenance":medium.provenance},
"readout":{"relative_time_edges_ns":edges.tolist()},
"events":[{"event_id":"course-shower","label":"Учебный ливень",
"sources":[{"type":"axis","position_m":source.frame.centre_m.tolist(),
"direction":source.frame.axis.tolist(),"extent_m":elements.extent_m}],
"time_origin_ns":origins.tolist(),"components":components.tolist(),
"charge_components":charge.tolist(),"diagnostics":{"elapsed_seconds":0.0,
"note":"Учебный axial-fast расчёт; знаковые бины сохранены."}}]}
out=Path(".build/course");out.mkdir(parents=True,exist_ok=True)
viewer=write_event_viewer(payload,out/"shower-viewer.html",json_path=out/"shower-viewer.json")
print(viewer)


### Задания

1. Удвоить M при неизменной сетке и отделить азимутальную ошибку.
2. Проверить знак демодуляции на единичном импульсе.
3. Сравнить сумму бинов с ω=0 и объяснить остаток конечного окна.
